# 09. 优化器与损失函数体系

很多初学者把“模型结构”当成深度学习的全部，但训练效果往往同样取决于两件事：

1. **损失函数**决定模型究竟在优化什么。
2. **优化器**决定模型如何沿着梯度更新参数。

同一个网络结构，换一个损失函数或优化器，训练行为可能完全不同。

## 先建立直觉

            优化器和损失函数可以分别理解成：

- 损失函数：告诉模型“什么叫做学得好”
- 优化器：告诉模型“应该怎么改参数才能变好”

很多人把模型结构当成主角，但在真实训练里，这两者几乎和结构同样重要。

例如：

- 你可以有一个很强的网络，但损失函数不适合，模型会学偏
- 你可以有一个合理的损失函数，但优化器不稳定，模型会收敛很慢甚至发散

## 架构是怎么工作的

            这一节虽然不讲“网络层结构”，但其实讲的是训练系统的底层架构。

训练闭环里有四个连续步骤：

1. 前向传播得到预测
2. 损失函数把预测变成一个可优化的标量
3. 反向传播得到梯度
4. 优化器根据梯度更新参数

这四步里，优化器和损失函数正好占了中间两环。

## 训练时到底发生了什么

            损失函数本质上定义了梯度长什么样；优化器本质上决定了你怎么使用这些梯度。

例如：

- MSE 会对大误差施加更强惩罚
- MAE 对异常值更稳
- Cross Entropy 会强烈推动正确类别概率上升
- Adam 会为不同参数自适应调整步长

所以“学得快不快、稳不稳、会不会过拟合”，并不只由网络结构决定。

## 什么时候该用它

            经验上可以这样选：

- 小中规模任务：Adam / AdamW 往往是很好的默认选项
- 视觉大模型后期微调：SGD + Momentum 仍很常见
- 回归含异常值：优先考虑 Huber
- 类别不平衡检测：可以考虑 Focal Loss

## 最常见的误区

            常见误区：

1. **误以为 Adam 永远最好**
   它通常收敛更快，但最终泛化不一定总赢。

2. **误以为损失下降更快就一定更好**
   有时只是优化得更快，不代表最终测试集更优。

3. **误以为 Cross Entropy 和 Accuracy 是同一件事**
   一个是训练目标，一个是评价指标，它们相关但不等价。

## 1. 常见优化器公式

### SGD

$$
\theta_{t+1} = \theta_t - \eta g_t
$$

其中 $g_t = \nabla_\theta \mathcal{L}(\theta_t)$，$\eta$ 是学习率。

### Momentum

$$
v_{t+1} = \beta v_t + g_t,\qquad \theta_{t+1} = \theta_t - \eta v_{t+1}
$$

作用：在一致方向上累积速度，降低震荡。

### RMSProp

$$
s_{t+1} = \rho s_t + (1-\rho) g_t^2,\qquad
\theta_{t+1} = \theta_t - \eta \frac{g_t}{\sqrt{s_{t+1}} + \epsilon}
$$

作用：为不同参数分配自适应步长。

### Adam

$$
m_{t+1} = \beta_1 m_t + (1-\beta_1) g_t
$$

$$
v_{t+1} = \beta_2 v_t + (1-\beta_2) g_t^2
$$

$$
\hat{m}_{t+1} = \frac{m_{t+1}}{1-\beta_1^{t+1}},\qquad
\hat{v}_{t+1} = \frac{v_{t+1}}{1-\beta_2^{t+1}}
$$

$$
\theta_{t+1} = \theta_t - \eta \frac{\hat{m}_{t+1}}{\sqrt{\hat{v}_{t+1}} + \epsilon}
$$

### AdamW

AdamW 和 Adam 的关键区别是：**权重衰减从梯度中解耦**。

$$
\theta_{t+1} = \theta_t - \eta \frac{\hat{m}_{t+1}}{\sqrt{\hat{v}_{t+1}} + \epsilon} - \eta \lambda \theta_t
$$

在现代 Transformer 训练中，AdamW 是非常常见的默认选择。

## 2. 常见损失函数公式

### 回归损失

- MSE
  $$
  \mathcal{L}_{\mathrm{MSE}} = \frac{1}{n}\sum_i (y_i - \hat{y}_i)^2
  $$
- MAE
  $$
  \mathcal{L}_{\mathrm{MAE}} = \frac{1}{n}\sum_i |y_i - \hat{y}_i|
  $$
- Huber
  $$
  \mathcal{L}_{\delta}(r)=
  \begin{cases}
  \frac{1}{2}r^2,& |r|\le \delta \\
  \delta(|r|-\frac{1}{2}\delta),& |r|>\delta
  \end{cases}
  $$

其中 $r = y - \hat{y}$。

### 分类损失

- Cross Entropy
  $$
  \mathcal{L}_{\mathrm{CE}} = -\sum_k y_k \log p_k
  $$
- Binary Cross Entropy
  $$
  \mathcal{L}_{\mathrm{BCE}} = -\left[y\log p + (1-y)\log(1-p)\right]
  $$
- Focal Loss
  $$
  \mathcal{L}_{\mathrm{Focal}} = -(1-p_t)^\gamma \log p_t
  $$

Focal Loss 会降低“简单样本”的权重，强调困难样本，因此常用于类别不平衡检测任务。

In [ ]:
# 兼容当前 Windows 环境：把临时目录固定到用户目录下的 ASCII 路径，
# 避免 scipy / sklearn 在中文工作目录下寻找临时文件时报错。
from pathlib import Path
import os
import warnings

temp_root = Path(os.environ.get("ML_DL_TMP", str(Path.home() / ".ml_dl_notebook_tmp")))
temp_root.mkdir(exist_ok=True)
os.environ["TMP"] = str(temp_root)
os.environ["TEMP"] = str(temp_root)

warnings.filterwarnings("ignore")

import math
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

np.random.seed(42)
random.seed(42)

sns.set_theme(style="whitegrid", context="talk")
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["axes.unicode_minus"] = False
plt.rcParams["font.sans-serif"] = ["Microsoft YaHei", "SimHei", "Arial Unicode MS", "DejaVu Sans"]


import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

torch.manual_seed(42)


print("临时目录:", temp_root)

In [ ]:
# 可视化不同回归损失对残差的惩罚形状。
residuals = np.linspace(-3, 3, 400)
mse = residuals ** 2
mae = np.abs(residuals)
delta = 1.0
huber = np.where(
    np.abs(residuals) <= delta,
    0.5 * residuals ** 2,
    delta * (np.abs(residuals) - 0.5 * delta),
)

plt.figure(figsize=(11, 6))
plt.plot(residuals, mse, label="MSE", linewidth=3)
plt.plot(residuals, mae, label="MAE", linewidth=3)
plt.plot(residuals, huber, label="Huber(delta=1.0)", linewidth=3)
plt.title("不同回归损失对残差的惩罚曲线")
plt.xlabel("残差 r = y - y_hat")
plt.ylabel("loss")
plt.legend()
plt.show()

In [ ]:
# 分类场景：比较交叉熵与 Focal Loss 对“置信度”的响应。
p = np.linspace(1e-4, 0.9999, 300)
ce = -np.log(p)
gamma = 2.0
focal = -(1 - p) ** gamma * np.log(p)

plt.figure(figsize=(11, 6))
plt.plot(p, ce, label="Cross Entropy", linewidth=3)
plt.plot(p, focal, label="Focal Loss (gamma=2)", linewidth=3)
plt.title("正样本预测概率 p 下的 CE / Focal 损失")
plt.xlabel("模型对正确类别的预测概率 p")
plt.ylabel("loss")
plt.legend()
plt.show()

## 3. 在同一任务上比较优化器

下面我们在 `digits` 数据集上训练相同的 MLP，仅仅替换优化器：

- SGD
- SGD + Momentum
- RMSprop
- Adam
- AdamW

这样可以更直观地看到“收敛速度”和“最终精度”的差异。

In [ ]:
from sklearn.datasets import load_digits
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

digits = load_digits()
X = digits.data.astype(np.float32)
y = digits.target.astype(np.int64)

X = StandardScaler().fit_transform(X).astype(np.float32)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

train_loader = DataLoader(
    TensorDataset(torch.tensor(X_train), torch.tensor(y_train)),
    batch_size=64,
    shuffle=True,
)
test_loader = DataLoader(
    TensorDataset(torch.tensor(X_test), torch.tensor(y_test)),
    batch_size=256,
    shuffle=False,
)


class OptimizerMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(64, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 10),
        )

    def forward(self, x):
        return self.net(x)


def build_optimizer(name, params):
    if name == "SGD":
        return torch.optim.SGD(params, lr=0.05)
    if name == "Momentum":
        return torch.optim.SGD(params, lr=0.03, momentum=0.9)
    if name == "RMSprop":
        return torch.optim.RMSprop(params, lr=0.001)
    if name == "Adam":
        return torch.optim.Adam(params, lr=0.001)
    if name == "AdamW":
        return torch.optim.AdamW(params, lr=0.001, weight_decay=1e-2)
    raise ValueError(name)


def train_with_optimizer(optimizer_name, epochs=15):
    model = OptimizerMLP()
    optimizer = build_optimizer(optimizer_name, model.parameters())
    criterion = nn.CrossEntropyLoss()
    history = {"train_loss": [], "test_acc": []}

    for _ in range(epochs):
        model.train()
        total_loss, total_count = 0.0, 0
        for batch_x, batch_y in train_loader:
            optimizer.zero_grad()
            logits = model(batch_x)
            loss = criterion(logits, batch_y)
            loss.backward()
            optimizer.step()

            total_loss += loss.item() * batch_x.size(0)
            total_count += batch_x.size(0)

        model.eval()
        all_preds = []
        with torch.no_grad():
            for batch_x, _ in test_loader:
                all_preds.append(model(batch_x).argmax(dim=1))

        preds = torch.cat(all_preds).numpy()
        history["train_loss"].append(total_loss / total_count)
        history["test_acc"].append(accuracy_score(y_test, preds))

    return model, history

In [ ]:
optimizer_names = ["SGD", "Momentum", "RMSprop", "Adam", "AdamW"]
opt_histories = {}
trained_opt_models = {}

for name in optimizer_names:
    model, history = train_with_optimizer(name, epochs=15)
    trained_opt_models[name] = model
    opt_histories[name] = history
    print(name, "最终测试准确率:", round(history["test_acc"][-1], 4))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for name, history in opt_histories.items():
    axes[0].plot(history["train_loss"], label=name)
    axes[1].plot(history["test_acc"], label=name)

axes[0].set_title("不同优化器的训练损失曲线")
axes[0].set_xlabel("epoch")
axes[0].set_ylabel("loss")
axes[0].legend()

axes[1].set_title("不同优化器的测试准确率曲线")
axes[1].set_xlabel("epoch")
axes[1].set_ylabel("accuracy")
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
optimizer_summary = pd.DataFrame(
    [
        {
            "优化器": name,
            "最终训练损失": history["train_loss"][-1],
            "最终测试准确率": history["test_acc"][-1],
        }
        for name, history in opt_histories.items()
    ]
).sort_values("最终测试准确率", ascending=False)

optimizer_summary

## 4. 什么时候用什么优化器 / 损失函数

### 优化器选择建议

- `SGD`：适合理解优化本质，也常见于大规模视觉训练的后期精调。
- `Momentum`：当损失面震荡明显时，比纯 SGD 更稳。
- `RMSprop`：历史上常用于 RNN 等训练不稳定场景。
- `Adam`：中小规模任务、快速原型、默认强基线。
- `AdamW`：现代 Transformer / ViT / LLM 训练中最常用。

### 损失函数选择建议

- `MSE`：误差服从高斯噪声、希望大误差被更强惩罚。
- `MAE`：对异常值更鲁棒，但梯度不如 MSE 平滑。
- `Huber`：想兼顾稳定性和鲁棒性时常用。
- `CrossEntropy`：多分类默认首选。
- `BCEWithLogitsLoss`：二分类、多标签分类默认首选。
- `Focal Loss`：类别极不平衡、困难样本稀少时有价值。